# Bước 1: Tải & Tiền Xử Lý LIDC-IDRI Lưu Trực Tiếp Vào Google Drive
### Mục tiêu: Tải từng ca CT -> Crop patch 128x128 int16 HU (Annotation 1) -> Lưu vào Drive -> Xóa DICOM ngay.

> **Lợi ích**:
> - Chỉ cần chạy bước này **MỘT LẦN DUY NHẤT**.
> - Toàn bộ dữ liệu sau xử lý chỉ nặng **~150MB** (thay vì 125GB thô), lưu vĩnh viễn trên Google Drive.
> - Ổ đĩa tạm Colab không bao giờ bị tràn (chỉ tốn < 500MB tại bất kỳ thời điểm nào).
> - Tự động bỏ qua (skip) các bệnh nhân đã xử lý nếu bị ngắt mạng giữa chừng.

### 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Thiết lập Codebase và Cài đặt Thư viện

In [ ]:
# Nếu tải lên mus_net.zip
!unzip -q /content/mus_net.zip -d /content/mus_net
%cd /content/mus_net

# Cài đặt các gói phụ thuộc (pylidc, requests, pyyaml...)
!pip install -q -r requirements.txt
!pip install -q requests

### 3. Thực hiện Tải, Tiền Xử Lý và Lưu Ngay Ra Drive
Dữ liệu huấn luyện (`.npz`) sẽ được lưu vào: `/content/drive/MyDrive/lidc_train_ready`

In [ ]:
DRIVE_TARGET = '/content/drive/MyDrive/lidc_train_ready'

# Chạy pipeline: tải từng ca, trích xuất nốt (int16 HU), lưu vào Drive, xóa DICOM ngay
# Mặc định xử lý toàn bộ 1018 ca (hoặc đặt --max_patients 50 để chạy thử nghiệm nhanh)
!python -m mus_net.data.prepare_lidc_to_drive \
  --drive_output_dir {DRIVE_TARGET} \
  --temp_dir /content/temp_dicom \
  --patch_size 128

### 4. Kiểm tra Manifest và Toàn Vẹn Dataset Trên Google Drive

In [ ]:
import json
from pathlib import Path

manifest_file = Path(DRIVE_TARGET) / 'dataset_manifest.json'
if manifest_file.exists():
    with open(manifest_file, 'r') as f:
        meta = json.load(f)
    print('=== THÔNG TIN DATASET TRÊN GOOGLE DRIVE ===')
    print(f"Tổng số lát cắt nốt  : {meta['total_slices']}")
    print(f"Tổng số bệnh nhân    : {meta['total_patients']}")
    print(f"Kích thước patch HU  : {meta['patch_shape']} ({meta['hu_data_type']})")
    print(f"Kích thước mask GT   : {meta['mask_shape']} ({meta['mask_data_type']})")
    print(f"Tiêu chuẩn chú thích : {meta['annotation_standard']}")
    print('\n-> Dataset đã sẵn sàng vĩnh viễn trên Drive cho các lần train tiếp theo!')
else:
    print('Chưa tìm thấy manifest. Vui lòng kiểm tra lại bước 3.')